## corner study
for the sake of time, we will not be looking at all corners over all the laps, instead we will try to take some corners that cover different types of corners for this track:
* C1 bc it's a wide fast
* [C2, C3, C4, C5] together bc they are esses
* C20 bc it's 90 degrees
* C24 and C27 bc hairpins
* C25 and C26 bc they're a snail corner

We will take the esses and the snail as one group and one window instead of two separate corners because it makes more sense.

and in the comparison, we will not take all these corners from all the laps, instead we will take:
* the corner of the reference lap (fastest lap)
* the fastest lap on that corner/group if it beats the reference one
* the slowest in stint 1
* the slowest in stint 2


In [24]:
from src.segment import load_segmented
from src.loader import load_centerline
from src.config import CORNER_GROUPS
from src.analysis import comparable_laps, reference_lap, corner_entries, group_times, pick_laps

In [25]:
data = load_segmented()
grid, laps, zones, apexes, gaps = data["grid"], data["laps"], data["zones"], data["apexes"], data["gaps"]
cl = load_centerline(); length = cl.attrs["length"]
laps_used = comparable_laps(laps)["lap"].tolist()
ref = reference_lap(laps)

entries = corner_entries(grid, apexes, zones, laps_used, length)
gt = group_times(grid, apexes, entries, zones, gaps, laps_used, CORNER_GROUPS, length)
picks = pick_laps(gt, ref)

In [26]:
picks

,group,role,lap,time,delta
0,C1,ref,10,4.083430,0.000000
1,C1,fastest,10,4.083430,0.000000
2,C1,slowest stint 1,3,4.930340,0.846910
3,C1,slowest stint 2,15,5.651795,1.568365
4,esses,ref,10,5.894141,0.000000
5,esses,fastest,22,5.686142,-0.207999
6,esses,slowest stint 1,3,6.541059,0.646918
7,esses,slowest stint 2,15,6.651147,0.757006
8,C20,ref,10,2.532906,0.000000
9,C20,fastest,21,2.355891,-0.177015


In [27]:
import numpy as np
import plotly.graph_objects as go
from plotly.colors import qualitative
from plotly.subplots import make_subplots
from src.segment import U_COL, THROTTLE_COL, BRAKE_COL, STEER_COL

CHANNELS = [(U_COL, "speed (m/s)"), (THROTTLE_COL, "throttle (0–1)"),
            (BRAKE_COL, "brake front (bar)"), (STEER_COL, "steering (deg, + = left)")]
COAST = "coasting" # phase label for both pedals off
main_apex = apexes[apexes["is_main"]].set_index("corner")["s"]


def plot_group(name: str) -> go.Figure:
    p = picks[picks["group"] == name]
    roles = p.groupby("lap", sort=False)["role"].agg(" + ".join)    # ref + fastest merged if same lap
    w = gt[gt["group"] == name].iloc[0]
    s0, s1 = float(w["s_start"]), float(w["s_end"])
    span = np.mod(s1 - s0, length)
    corners = CORNER_GROUPS[name]

    fig = make_subplots(rows=len(CHANNELS), cols=1, shared_xaxes=True, vertical_spacing=0.03)
    for k, (lap, role) in enumerate(roles.items()):
        lg = grid[grid["lap"] == lap].copy()
        lg["d"] = np.mod(lg["s"] - s0, length)                       # metres from window start
        lg = lg[lg["d"] <= span].sort_values("d").reset_index(drop=True)
        color = qualitative.Plotly[k % len(qualitative.Plotly)]
        label = f"lap {lap} ({role})"

        coast = (lg["phase"] == COAST).to_numpy()
        coast_ext = coast | np.roll(coast, 1) | np.roll(coast, -1)   # overlap one point so lines join
        lap_entries = entries[(entries["lap"] == lap) & (entries["corner"].isin(corners))]

        for i, (col, ylabel) in enumerate(CHANNELS, start=1):
            y = lg[col].to_numpy()
            fig.add_trace(go.Scatter(x=lg["d"], y=np.where(coast, np.nan, y), mode="lines",
                                     line=dict(color=color), name=label, legendgroup=str(lap),
                                     showlegend=(i == 1)), row=i, col=1)
            fig.add_trace(go.Scatter(x=lg["d"], y=np.where(coast_ext, y, np.nan), mode="lines",
                                     line=dict(color=color, dash="dash"), name=f"{label} coasting",
                                     legendgroup=str(lap), showlegend=False), row=i, col=1)

            for _, e in lap_entries.iterrows():                      # braking point / corner entry
                xe = float(np.mod(e["s_entry"] - s0, length))
                fig.add_trace(go.Scatter(x=[xe], y=[np.interp(xe, lg["d"], y)], mode="markers",
                                         marker=dict(color=color, size=11, symbol="triangle-down",
                                                     line=dict(color="black", width=1)),
                                         name=label, legendgroup=str(lap), showlegend=False,
                                         hovertext=f"lap {lap}: C{int(e['corner'])} entry ({e['entry_source']})",
                                         hoverinfo="text"), row=i, col=1)
            fig.update_yaxes(title_text=ylabel, row=i, col=1)

        u_exit = float(lg[U_COL].iloc[-1])                           # exit speed at window end
        fig.add_trace(go.Scatter(x=[float(lg["d"].iloc[-1])], y=[u_exit], mode="markers",
                                 marker=dict(color=color, size=12, symbol="diamond",
                                             line=dict(color="black", width=1)),
                                 name=label, legendgroup=str(lap), showlegend=False,
                                 hovertext=f"lap {lap}: exit {u_exit:.2f} m/s", hoverinfo="text"),
                      row=1, col=1)

    for c in corners:                                                # main apexes of the group
        x = float(np.mod(main_apex[c] - s0, length))
        fig.add_vline(x=x, line_dash="dot", line_color="grey")
        fig.add_annotation(x=x, y=1.0, yref="paper", text=f"C{c} apex", showarrow=False, yanchor="bottom")

    # legend keys for the symbols
    for sym, txt in [("triangle-down", "entry / braking point"), ("diamond", "exit speed")]:
        fig.add_trace(go.Scatter(x=[None], y=[None], mode="markers", name=txt,
                                 marker=dict(color="grey", size=11, symbol=sym)), row=1, col=1)
    fig.add_trace(go.Scatter(x=[None], y=[None], mode="lines", name="coasting",
                             line=dict(color="grey", dash="dash")), row=1, col=1)

    fig.update_xaxes(title_text="metres from window start", row=len(CHANNELS), col=1)
    fig.update_layout(title=f"{name}: picked laps", height=1050, width=1300,
                      legend=dict(orientation="h", y=1.07))
    return fig


In [28]:
# group C1, wide 
plot_group("C1").show()

Comparison of laps 3 and 15 with lap 10:
* Lap 15: there doesn't seem to be any braking point; the driver instead opted to lift at around 16.5 m. The driver was already entering the corner slowly, at 8.7 m/s and rising, and lifted when he reached around 11 m/s. After the apex the driver presses back on the throttle very little, and the speed picks up very gradually until he exits the corner at 13.85 m/s.
* Lap 3 on the other hand, brakes, and does it much earlier, at around 2.7 m from the start of the window. We notice that the driver coasts before hitting the brakes, so most likely they started lifting, **coasted for 1.5 m** and then hit the brakes, all this at 12.7 m/s. The transition between lifting and braking was not smooth: we expect the driver to lift off the throttle and press the brakes gradually, whereas here it looks more like they lifted off completely and then started braking. The braking pattern does not seem smooth at all either. We usually expect the trace to go up and then back down once, but instead it went up, then down, then up again and then down, so it seems like the driver was hesitant. The driver then lifts off the brakes at 12.7 m from the window start, at 11 m/s. The **coasting lasts about 10 m**, then the driver touches the throttle a tiny bit right before the apex and starts actually accelerating, with an upward trend, at 26.1 m (after the apex). The max throttle reaches 0.61, at a speed of 9.8 m/s, which then increases gradually, and he exits at 19.33 m/s.
* Lap 10: the driver enters at 15.2 m/s with the throttle at 0.73. He then starts lifting off the throttle and braking at the same time, gradually and smoothly, and starts to turn the car as well. The max braking reaches 9.5 bar and the throttle reaches 0. The braking lasts 13 m, with **1 m of coasting between ~16.4 and ~17.4 m**. The minimum speed is 12.2 m/s and is reached before the apex, and the speed is maintained through the corner's apex. The driver starts pressing the throttle gradually before the apex, at 18.6 m, and it goes up to 0.6 at 30 m. Then the driver lifts a little and goes back on the throttle at 37.8 m until it reaches 0.78, and it stays fairly steady with the speed rising until he exits at 21.59 m/s. The hesitation here on the throttle might have cost him some time, though.

* Additional notes:
  - A lot of the loss comes from C27, before C1 starts.

In [29]:
from src.analysis import corner_metrics
import pandas as pd
cm = corner_metrics(grid, entries, apexes, zones, laps_used, length)

# positions are metres from apex, 0 before apex and + after apex
SHOW = ["entry_from_apex", "peak_brake_bar", "brake_total_time", "coasting_time", "coasting_dist",
        "min_speed", "min_speed_from_apex", "pickup_from_apex", "max_exit_throttle", "exit_speed",
        "steer_corrections", "exit_slip_pct", "corner_time"]


def group_metrics(name: str) -> pd.DataFrame:
    p = picks[picks["group"] == name].drop_duplicates("lap")
    lap_list = p["lap"].tolist()
    m = cm[cm["lap"].isin(lap_list) & cm["corner"].isin(CORNER_GROUPS[name])]
    out = m.set_index(["corner", "lap"])[SHOW].T 

    s0 = float(gt.loc[gt["group"] == name, "s_start"].iloc[0])
    for lap in lap_list:   
        lg = grid[grid["lap"] == lap].sort_values("s")
        u0 = float(np.interp(s0, lg["s"], lg[U_COL]))
        r = p[p["lap"] == lap].iloc[0]
        for c in CORNER_GROUPS[name]:
            out.loc["speed_at_window_start", (c, lap)] = u0
            out.loc["group_time", (c, lap)] = r["time"]
            out.loc["group_delta_vs_ref", (c, lap)] = r["delta"]

    for c in CORNER_GROUPS[name]:
        print(f"C{c} apex at {np.mod(main_apex[c] - s0, length):.1f} m from window start")
    return out.round(2)

remember that from notebook 8:
**notice that the metric pickup_from_apex has NaN values, this could be one of three reasons, either the throttle was back on AFTER the corner zone end, the throttle was on the whole time so it never went off, or the throttle was never on anywhere in teh window.**

In [30]:
group_metrics("C1")

C1 apex at 24.7 m from window start


corner                     1              
lap                       3      10     15
entry_from_apex       -22.00 -21.40  -7.90
peak_brake_bar          3.73   9.55   0.00
brake_total_time        0.81   0.85   0.00
coasting_time           0.92   0.04   0.00
coasting_dist          10.50   0.60   0.00
min_speed               9.85  12.20  10.83
min_speed_from_apex     2.00   0.20  -5.80
pickup_from_apex       -2.20  -6.70    NaN
max_exit_throttle       0.62   0.78   0.23
exit_speed             19.33  21.59  13.85
steer_corrections       2.00   4.00   1.00
exit_slip_pct           1.82  19.79  -1.01
corner_time             4.73   3.87   4.00
speed_at_window_start  12.72  15.20   8.72
group_time              4.93   4.08   5.65
group_delta_vs_ref      0.85   0.00   1.57

in the case of lap 15, the NaN in pickup_from_apex happened is because the throttle never went off.

* **Metrics:**
  - The biggest difference between lap 3 and lap 10 is the coasting: lap 3 coasts for 0.92 s over 10.5 m, while lap 10 coasts for only 0.04 s over 0.6 m.
  - Lap 3 actually brakes for about as long as lap 10 (0.81 s vs 0.85 s), but with much less pressure (3.7 bar vs 9.5 bar). So the driver is not braking less, he is braking lighter and for longer, which matches the hesitation we see in the braking trace.
  - On lap 3 the minimum speed is reached 2 m after the apex, while on lap 10 it is reached right at the apex (0.2 m).
  - The throttle lift on lap 10 after the apex is most likely not hesitation: the rear wheels were slipping 19.8% at the exit and the driver made 4 steering corrections, against about 2% and 2 corrections on lap 3. So the rear was most likely sliding, and the driver lifted and corrected the steering to catch it. A smoother throttle pickup could make lap 10 even faster here.
  - The minimum speed is not the whole story: lap 15 has a higher minimum speed than lap 3 (10.8 m/s vs 9.9 m/s) but is still 0.7 s slower. What costs lap 15 the most is the slow entry (8.7 m/s) and the low exit speed (13.9 m/s), with the throttle never going above 0.23.
  - The time loss is measured on the same window for all laps: lap 3 loses 0.85 s and lap 15 loses 1.57 s compared to lap 10. (The corner time from each lap's own entry can't be compared here, because lap 15's entry is only 7.9 m before the apex, which makes its corner time look almost as good as lap 10's.)
  - Both lap 3 and lap 15 arrive at C1 slower than lap 10 (12.7 m/s and 8.7 m/s vs 15.2 m/s), and that carries into the corner. The time lost in C27 itself is not counted here; we will see how much it costs when we look at C27.

In [31]:
#  the esses
plot_group("esses").show()

Comparison of laps 22, 3 and 15 with lap 10 in the esses:
* Lap 10 (ref): the driver enters the fastest of all, at 22.85 m/s with the throttle at 0.57, and lifts off the throttle and starts braking at the same time at around 2 m. The braking is very hard, it goes up to 16 bar at around 11 m and stays between 12 and 15 bar until around 34 m, then the driver releases the brakes right before the C2 apex. The speed drops to around 9 m/s at the C2 apex. The driver then goes back on the throttle at the C2 apex up to 0.2, but lifts off again at around 43 m and brakes a second time for C3 (around 4.8 bar) before going back on the throttle at around 46 m. So between C2 and C3 the driver goes throttle, brake, throttle, which looks like he slowed down too much for C2 and then had to correct for C3. The minimum speed is around 8.1 m/s between C2 and C3. Between C3 and C4 the throttle drops again to around 0.1 while the steering is at full lock, then the driver gets back on the throttle from C4 up to 0.65 before C5 and exits the fastest of all, at 15.5 m/s.
* Lap 22 (fastest): the driver enters slower than lap 10, at 20.3 m/s, and is already coasting at the start of the window. He brakes at around 2.5 m but the braking builds up gradually and is much lighter than lap 10, around 9.5 bar at its peak and around 7 to 8.5 bar until he releases at the C2 apex. Even though he brakes less, he carries more speed into C2, around 10.7 m/s at the apex against 9 m/s for lap 10. He then coasts through the change of direction between C2 and C3 and does not brake at all for C3, and he also turns earlier and more than the others (the steering goes down to around −80deg). He stays the fastest through the middle of the esses with the speed between 9.3 and 10.3 m/s. Between C3 and C4 the throttle also drops to around 0.12 with a short coast before C4, and then goes up to 0.48 and stays steady until the exit. He exits slower than lap 10, at around 13.9 m/s, so his time is gained in the middle of the esses, not at the exit.
* Lap 3: the driver enters at 20.8 m/s with the throttle at 0.43, and starts braking at around 1.5 m. The braking goes up to around 11 bar and stays at around 10 bar until 33 m, but then the driver does not release before the apex like the others, the brake pressure goes down slowly and he is still braking at 5 to 6 bar after the C2 apex until around 41 m. Because of that he is the slowest through the middle, around 7.5 to 8 m/s between C2 and C3. He also holds the steering for C2 longer and changes direction later than the others, with some coasting during the change. He gets on the throttle after C3 up to 0.3, then lifts off completely and coasts for a few metres before C4, then gets back on the throttle gradually up to 0.5 and exits at around 13.6 m/s.
* Lap 15: the driver enters very slowly, at 14.2 m/s, and is coasting from the start of the window. He coasts for around 24 m and only brakes lightly at around 27 m, with a peak of around 3.7 bar. Since he entered so slowly, by the time the other drivers finish braking he is at the same speed as them, around 10.6 m/s at the C2 apex. He then brakes again for C3 at around 4 bar, for longer than lap 10, and gets on the throttle after C3 but only up to 0.22. He changes direction later than the others between C3 and C4, then gets back on the throttle up to around 0.4 but the throttle goes down again towards the exit, and he exits the slowest of all at around 12.6 m/s.

* **Additional notes:**
  - The fastest lap through the esses is not the one that brakes the hardest. Lap 22 brakes lighter, carries more speed into C2 and flows through C2 and C3 without braking again, while lap 10 brakes very hard, slows down too much and has to brake again for C3.
  - Lap 10 has the best exit, so the best way through the esses would be the middle of lap 22 with the exit of lap 10.
  - All the laps lift or drop the throttle between C3 and C4, so it looks like that part of the esses is hard to drive on the throttle for both drivers.
  - Lap 15 shows the same style as in C1, a long coast on the approach, light braking, and low throttle on the exit.

In [32]:
group_metrics("esses")

C2 apex at 37.1 m from window start
C3 apex at 49.1 m from window start
C4 apex at 59.1 m from window start
C5 apex at 69.1 m from window start


corner,2,3,4,5,2,3,4,5,2,3,4,5,2,3,4,5
lap,3,3,3,3,10,10,10,10,15,15,15,15,22,22,22,22
entry_from_apex,-35.60,-5.00,-8.40,-3.00,-35.30,-7.70,-7.50,-3.00,-10.10,-5.00,-6.30,-3.00,-34.70,-5.00,-5.70,-3.00
peak_brake_bar,11.48,0.34,0.00,0.00,16.08,4.87,0.04,0.04,3.77,4.51,0.00,0.00,9.55,0.00,0.01,0.00
brake_total_time,2.11,0.00,0.00,0.00,1.92,0.42,0.00,0.00,0.76,0.21,0.00,0.00,2.08,0.00,0.00,0.00
coasting_time,0.00,0.41,0.27,0.00,0.00,0.00,0.00,0.00,1.73,0.10,0.00,0.00,0.10,0.00,0.00,0.00
coasting_dist,0.00,2.40,2.70,0.00,0.00,0.00,0.00,0.00,24.00,0.90,0.00,0.00,2.40,0.00,0.00,0.00
min_speed,8.75,7.33,8.17,10.09,9.11,8.07,9.17,11.87,10.61,8.26,8.57,10.70,10.60,9.29,10.06,11.45
min_speed_from_apex,0.40,-2.30,-2.10,-2.80,-1.70,-2.90,-7.50,-2.80,0.40,-0.50,-1.80,-2.80,-0.50,-0.80,-5.70,-2.80
pickup_from_apex,NaN,-2.60,-3.00,NaN,-0.80,-3.20,NaN,NaN,NaN,-2.00,NaN,NaN,NaN,NaN,NaN,-2.20
max_exit_throttle,0.00,0.28,0.45,0.49,0.08,0.32,0.64,0.65,0.00,0.18,0.42,0.37,0.00,0.26,0.42,0.49


* **Metrics:**
  - Lap 22 brakes for about as long as lap 10 in C2 (2.08 s vs 1.92 s) but much lighter (9.55 bar vs 16.08 bar), and it comes out of C2 at 10.6 m/s against 9.1 m/s for lap 10. So the fastest lap is not braking less, it is braking lighter and carrying more speed into the corner.
  - Lap 10 is the only fast lap that brakes again for C3 (4.87 bar), lap 22 does not brake at all there. The minimum speed between C2 and C3 is 8.07 m/s for lap 10 against 9.29 m/s for lap 22, which shows how much speed lap 10 loses by slowing down too much in C2.
  - Lap 3 brakes the longest of all in C2 (2.11 s) and its minimum speed comes 0.4 m after the C2 apex, which matches the late brake release. It is also the slowest between C2 and C3 with a minimum speed of 7.33 m/s, and it coasts for 2.4 m in C3 and 2.7 m in C4, which is the coast before C4 that we see on the plot.
  - Lap 15 coasts for 24 m (1.73 s) before braking for C2, but its minimum speed in C2 (10.61 m/s) is the same as lap 22. So the coasting does not cost him speed at the apex, it costs him time on the approach, where he is much slower than everyone.
  - At the exit of the esses lap 10 is the fastest with 15.48 m/s, against 13.89 m/s for lap 22, 13.39 m/s for lap 3 and 12.54 m/s for lap 15.
  - Lap 10 has a very high rear wheel slip at the C5 exit (48%), and also 12.9% in C3. That is a lot of wheelspin, so either the driver is putting too much throttle on the exit, or this measurement is off for this lap. This needs to be checked before we use it.
  - Corrections to the plot reading: lap 3 exits at 13.39 m/s, not 13.6 m/s. Lap 22 does not really coast between C2 and C3, it barely touches the throttle (around 0.02), so it is only 2.4 m of coasting in C2.

In [33]:
# 90 degrees corner
plot_group("C20").show()

Lap 21's data is not reliable. It is already braking at around 4 bar at the start of the window and there is a break in the brake trace at around 16 to 18 m, so its entry point at 18 m is not its real braking point. It also enters around 4 m/s slower than lap 10 and still comes out as the fastest, which is not possible. This is because the position data (s) of lap 21 is broken in this part of the track, after a gap in the data the position catches up quickly and the time in the window comes out too short. So lap 21 will be left out.

In [34]:
gt.loc[(gt["group"] == "C20") & (gt["lap"] == 21), "gap"] = True   # s broken on lap 21 around C20 (catch-up after a NaN run)
picks = pick_laps(gt, ref)
plot_group("C20").show()

Comparison of laps 4, 8and 17 with lap 10 in C20:
* Lap 10 (ref): the driver enters at 20.5 m/s and is still accelerating, the throttle goes up to 0.58 and the speed reaches 21.2 m/s at around 5 m. He then lifts and starts braking at the same time at 5 m, so the transition from throttle to brake is smooth with a short overlap. The braking builds up quickly to 18 bar at around 9.5 m and stays between 17 and 19 bar until around 20 m, then he reduces it to around 10 bar and releases at around 34 m. The steering shows that he turns in early, around 21 m, up to -33 deg, then almost straightens the wheel again at around 28 m and turns in a second time up to -90 deg at 35 m. So he turns in twice, which looks like he turned too early and had to correct. He touches the throttle a little (0.14) at around 32 m while still braking lightly, then keeps it very low (around 0.06) through the apex. He carries the highest speed at the apex, around 8.3 m/s, and exits at around 8 m/s.
* Lap 8 (fastest): the driver enters at 21.3 m/s with the throttle at 0.81, the highest of all, and the speed is still rising to around 21.9 m/s at 2.5 m. He lifts off the throttle quickly and starts braking at 2.4 m, which is the earliest braking point of all, with a short overlap between throttle and brake until around 6 m. The braking builds up quickly to 13 bar at 6 m and around 17 to 18.5 bar between 10 and 19 m, then he reduces it to around 10 to 12 bar between 25 and 30 m and releases at around 35 m. Unlike lap 10 he does not turn in early. The steering stays straight until around 29 m and then he turns in once, smoothly, to around -96 deg at 35 m. Because he keeps the car straight while braking, he carries a bit more speed than lap 10 between 20 and 28 m, which is exactly where lap 10 turns in the first time. He touches the throttle very lightly (around 0.05) from 33 m while still braking a little, then picks it up after the apex to 0.27. His speed at the apex is lower than lap 10 (around 7.4 m/s against 8.3 m/s) and he exits at around 7.7 m/s, so his time is gained in the braking zone and not at the apex.
* Lap 4: the driver enters at 20.3 m/s and lifts off the throttle slowly from 0.39 to 0 by around 4.5 m, and starts braking at 3.9 m. The braking is not smooth, it goes up to 9.2 bar at 7 m, drops back to 6 bar at around 9.5 m and then goes up again to 17.4 bar at 15 m, like the hesitation we saw on lap 3 in C1. He keeps around 16 to 17 bar until 22 m, then reduces to around 10 bar and releases at around 35 m. He turns in the latest of all, the steering stays close to 0 until around 30 m and then goes quickly to -95 deg at 35.5 m. He coasts for a couple of metres before the apex and has the lowest speed of all, around 7 m/s, then gets on the throttle quickly up to 0.35 and exits at around 7.8 m/s.
* Lap 17: the driver enters slowly, at 15.8 m/s, and is coasting from the start of the window for around 17 m without any throttle. He brakes late, at 16.8 m, and the braking builds up gradually to around 14.7 bar at 26.5 m, then releases at around 34 m. Since he entered slowly, he is at the same speed as the others when he starts braking, and he carries a bit more speed than them through the braking zone. He turns in late, at around 29 m, up to around -98 deg, coasts for a few metres before the apex, then gets on the throttle up to 0.34 and exits the fastest of the three at around 8.5 m/s.

* **Additional notes:**
  - The exit point of C20 is only 2 m after the apex because the braking for C21 starts right after, so the exit speed here is almost the apex speed. The acceleration out of C20 is not really measured in this window.
  - Lap 17 shows the same style as lap 15 in C1 and the esses, a long coast on the approach and a slow entry, but here the braking itself is late and fairly firm.
  - Lap 8 brakes in a straight line and turns in once, while lap 10 turns in early, straightens and turns in again. Lap 8 carries more speed in the braking zone and lap 10 carries more at the apex, and in the end they are almost equal (0.04 s), so the double turn-in does not cost lap 10 much here.

In [35]:
group_metrics("C20")

C20 apex at 38.2 m from window start


corner                    20                     
lap                       4      8      10     17
entry_from_apex       -34.30 -35.80 -33.10 -21.40
peak_brake_bar         17.38  18.50  19.22  14.62
brake_total_time        2.16   2.08   2.07   1.28
coasting_time           0.26   0.00   0.00   2.05
coasting_dist           1.50   0.00   0.00  31.20
min_speed               6.99   7.63   7.96   7.54
min_speed_from_apex    -1.30   1.40   2.00  -0.70
pickup_from_apex       -1.60  -4.30  -6.40  -1.60
max_exit_throttle       0.35   0.28   0.14   0.34
exit_speed              7.88   7.68   7.96   8.53
steer_corrections       3.00   1.00   3.00   1.00
exit_slip_pct           1.12   7.26  12.97  29.90
corner_time             3.01   2.38   2.29   2.17
speed_at_window_start  20.31  21.33  20.44  15.80
group_time              3.21   2.49   2.53   3.23
group_delta_vs_ref      0.68  -0.04   0.00   0.70

* **Metrics:**
  - Lap 8 is only 0.04 s faster than lap 10 in C20, so the two laps are basically equal here. Lap 8 brakes the earliest (35.8 m before the apex) and both brake for the same time (2.08 s and 2.07 s), with lap 10 braking a bit harder (19.22 bar against 18.5 bar).
  - The steering corrections back up the double turn-in on lap 10, 3 corrections against 1 for lap 8. Lap 10 still has the highest minimum speed (7.96 m/s against 7.63 m/s for lap 8), so the speed it loses in the braking zone it gets back at the apex.
  - Lap 10 picks up the throttle the earliest (6.4 m before the apex) but uses the least of it, 0.14 at the most against 0.28 for lap 8.
  - Lap 4 loses 0.68 s. It has the lowest minimum speed of all (6.99 m/s), 3 steering corrections and 1.5 m of coasting before the apex, which matches the hesitant braking and the late turn-in we see on the plot.
  - Lap 17 loses 0.70 s and coasts for 31.2 m (2.05 s) in total, which is longer than what we see in the window because the coasting already starts before it, right after the throttle is released out of C19. It brakes the lightest (14.62 bar) and for the shortest time (1.28 s), and exits the fastest of all at 8.53 m/s, so like in C1 and the esses the time is lost on the approach and not in the corner itself.
  - Lap 17 also has a high rear wheel slip at the exit (29.9%), with the throttle at 0.34.
  - Corrections to the plot reading: the minimum speed of lap 10 is 7.96 m/s (not 8.3 m/s) and of lap 8 is 7.63 m/s (not 7.4 m/s), both reached just after the apex.

In [36]:
# hairpin
plot_group("C24").show()

Comparison of laps 7, 3 and 16 with lap 10 in C24:
* Lap 10 (ref): the driver enters at 18.6 m/s with the throttle at 0.77 and the speed is still rising to around 20.2 m/s at 5 m. He lifts off the throttle gradually from around 3.5 m and starts braking at 5.1 m, so there is an overlap between throttle and brake until around 8 m. The braking builds up gradually to around 18 bar at 12 m and stays between 16 and 18 bar until around 26 m, then he reduces it and releases at around 32 m. He turns in at around 23 m up to around 110 deg at 30 m, which is almost full lock, while he is still braking at around 10 bar. At that point the speed drops a lot, down to around 4.4 m/s at 30.5 m, much lower than all the other laps (around 6 m/s). So it looks like he braked too long into the corner with full lock and slowed the car down too much. He coasts for a few metres after releasing the brakes, then gets on the throttle quickly at around 35 m up to 0.4 and keeps it steady. He unwinds the steering quickly right after the apex and exits at around 8.2 m/s.
* Lap 7 (fastest): the driver enters at 19.2 m/s with the throttle at 0.83 and starts braking at 3.9 m. The braking goes up very quickly to 18.5 bar at around 8 m and stays there until around 15.5 m, then he releases a lot of pressure quickly down to around 8 bar and modulates the brake after that, going up and down between 8 and 14 bar until he releases at around 33 m. He turns in the earliest of all, at around 22.5 m, but only up to around 42 deg and holds it until around 31 m. Then the steering goes the other way, down to around -48 deg at 35 m, which looks like the rear of the car stepped out and he had to counter steer to catch it, before turning back to around 95 deg at 38 m. His minimum speed is around 6 m/s, he gets on the throttle at around 33 m while he is still catching the car, and he uses the most throttle of all on the exit, up to 0.55. He exits at around 8.3 m/s.
* Lap 3: the driver enters slower, at 17.4 m/s, with the throttle at 0.64, and starts braking at 3.6 m. The braking is lighter and not smooth again, it goes up to around 12.7 bar at 10 m, drops to around 10 bar, then drops a lot to around 4.5 bar at 19 m and goes back up to around 11 bar, before reducing and releasing at around 34.5 m. He turns in the latest of the stint 1 laps, at around 26 m, up to around 108 deg at 33 m. Because he brakes lighter he carries more speed than the others between 27 and 32 m, but then the speed drops to around 6.5 m/s and he coasts for a couple of metres before getting on the throttle at around 35 m. The throttle goes up to around 0.28, drops to around 0.17 at the apex and then goes back up to 0.33, and he exits at around 7.4 m/s.
* Lap 16: the driver enters slowly, at 15.3 m/s, lifts off the throttle at the start of the window and coasts for around 6 m before braking at 7.5 m. The braking builds up very gradually, it only reaches around 11 bar at 24 m, then he reduces it and releases at around 31.5 m. He turns in at around 24.5 m up to around 103 deg, then opens the steering before the apex, down to around 45 deg at the apex. He gets on the throttle the earliest of all, at around 31 m, up to 0.34, but then lifts to around 0.19 just before the apex and gets back on it to 0.32. The speed rises to around 7.8 m/s before the apex and then drops again, and he exits the slowest of all at around 7.2 m/s.

* Additional notes:
  - The fastest lap brakes the hardest at the start and then modulates the brake, and the car seems to rotate (with the counter steer) before the exit, which lets the driver get on the throttle harder than everyone else.
  - Lap 10 is the only lap that drops to around 4.4 m/s in the middle of the corner, while still braking at full lock.
  - Lap 3 and lap 16 show the same style as in the other corners. Lap 3 has light and uneven braking, and lap 16 enters slowly with a coast on the approach and a very gradual braking.

In [37]:
group_metrics("C24")

C24 apex at 38.6 m from window start


corner                    24                     
lap                       3      7      10     16
entry_from_apex       -35.00 -34.70 -33.50 -31.10
peak_brake_bar         12.86  18.59  18.40  11.22
brake_total_time        2.23   1.92   1.73   1.90
coasting_time           0.18   0.00   0.39   0.34
coasting_dist           0.90   0.00   2.70   5.40
min_speed               6.35   5.97   4.39   6.57
min_speed_from_apex    -3.20  -5.30  -8.60  -6.20
pickup_from_apex       -3.80  -5.90  -3.80  -7.40
max_exit_throttle       0.34   0.56   0.41   0.33
exit_speed              7.48   8.08   8.24   7.25
steer_corrections       1.00   3.00   1.00   3.00
exit_slip_pct           5.56  65.33  67.74  13.58
corner_time             3.71   3.12   3.19   3.22
speed_at_window_start  17.48  19.23  18.62  15.38
group_time              3.92   3.32   3.51   3.70
group_delta_vs_ref      0.41  -0.20   0.00   0.19

* Metrics:
  - Lap 7 is 0.20 s faster than lap 10 in C24. It brakes the hardest (18.59 bar) with no coasting at all, and it has 3 steering corrections, which matches the counter steer we see on the plot. It also uses the most throttle on the exit (0.56) and picks it up early, 5.9 m before the apex.
  - Lap 10 has the lowest minimum speed of all by far, 4.39 m/s at 8.6 m before the apex, against around 6 to 6.6 m/s for the other laps. It also coasts for 2.7 m after releasing the brakes. Even so it only loses 0.20 s to lap 7 and has the highest exit speed (8.24 m/s), so it gets back a lot of the time on the exit.
  - Both fast laps have a very high rear wheel slip on the exit, 67.7% for lap 10 and 65.3% for lap 7, against 13.6% for lap 16 and 5.6% for lap 3. The slip goes up with the throttle (0.41 and 0.56 for the fast laps, around 0.33 for the slow ones), so it looks like the fast laps get on the throttle hard at low speed and the rear wheels spin. The slip here is only measured over the 2 m between the apex and the exit point, so the number is rough, but the difference between the laps is clear. Less wheelspin could mean an even better exit for both fast laps.
  - Lap 3 loses 0.41 s. It brakes the longest (2.23 s) but the lightest of the stint 1 laps (12.86 bar), which matches the uneven braking on the plot, and it has the lowest exit throttle with lap 16 (0.34).
  - Lap 16 loses 0.19 s, less than lap 3. It coasts for 5.4 m before braking, brakes the lightest (11.22 bar), and picks up the throttle the earliest (7.4 m before the apex) but only up to 0.33, and it exits the slowest at 7.25 m/s.
  - Correction to the plot reading: lap 7 exits at 8.08 m/s, not 8.3 m/s, so lap 10 has the highest exit speed.

In [38]:
# snail corner
plot_group("snail")

Comparison of laps 23, 5 and 21 with lap 10 in the snail:
* Lap 10 (ref): the driver enters at 8.2 m/s with the throttle at around 0.4 and lifts a little from around 5 m to 0.27 through the C25 apex, but never lifts off completely. He carries the highest speed at the C25 apex, around 10.1 m/s, then gets back on the throttle up to 0.5 at 14 m and the speed keeps rising to around 12.1 m/s at 22.5 m. He then lifts off and brakes at around 22.6 m, up to around 6.3 bar, and releases at around 31 m. After the braking he stays on the throttle very lightly (around 0.07 to 0.14) between 32 and 42 m, then coasts for a few metres before the C26 apex and gets back on the throttle at around 46 m up to 0.3. His speed at the C26 apex is the lowest of all, around 8.4 m/s, and he exits at around 9.1 m/s.
* Lap 23 (fastest): the driver enters at 7.7 m/s with the throttle at 0.61, the highest of all, but lifts sharply to almost 0 at around 5.5 m before C25 and gets back on it right after up to 0.45. He is slower than lap 10 at the C25 apex (around 9.4 m/s) but catches up by 20 m. He lifts off earlier than the others, at around 19.5 m, coasts for about 2 m and brakes at 21.6 m. His braking is the shortest and the lightest of the three that brake properly, up to around 5.7 bar, and he releases at around 27.5 m. Because of that he carries more speed than lap 10 after the braking (around 10.4 m/s at 30 m against 9.8 m/s). He then coasts and stays almost off the throttle (around 0.05) until C26, and his speed at the C26 apex is higher than lap 10 (around 8.8 m/s). But he is slow to get back on the throttle after C26, it only goes up to 0.24 at the exit, and he exits the slowest of all at around 8.55 m/s.
* Lap 5: the driver enters the slowest, at 7.3 m/s, and lifts off the throttle completely before C25 with a short coast, so he is the slowest through C25 (around 8.6 m/s at the apex). He then stays on the throttle at 0.44 the longest of all, until around 23 m, and reaches the highest speed of all, around 12.4 m/s. Because of that he brakes the latest (23.7 m) and the hardest (around 7.1 bar), and after releasing at around 31 m he keeps dragging the brake lightly (around 0.5 to 1 bar) until around 36 m. He gets on the throttle up to around 0.24 between 37 and 40 m, then coasts before C26, and gets back on the throttle quickly at around 46 m. He exits the fastest of all at around 9.45 m/s.
* Lap 21: the driver enters at 7.5 m/s and stays on a low and steady throttle (around 0.25 to 0.29) all the way through C25 and up to 20 m, and he is the slowest of all through this part (around 10.5 m/s at the most). He lifts at around 20.5 m and brakes very lightly (around 2.3 bar) until around 26.5 m. After that he coasts a lot, the speed keeps dropping to around 8.25 m/s at 44 m, and he only gets back on the throttle at around 43 m, up to around 0.22. At the very end he lifts and touches the brake again, and he exits at around 8.85 m/s.

* Additional notes:
  - The fastest lap is not the fastest at C25 or at the exit. It gains time in the middle of the snail by lifting earlier, braking shorter and lighter, and carrying more speed after the braking towards C26.
  - Lap 5 shows the opposite, it stays on the throttle the longest and reaches the highest speed, but then has to brake the latest and the hardest and keeps dragging the brake, so the extra speed is lost again.
  - All the laps coast or barely use the throttle between the braking and C26, like between C3 and C4 in the esses.
  - Lap 21 is the lap that had broken position data around C20, so it is worth checking that its numbers look normal here before we trust them.

In [39]:
group_metrics("snail")

C25 apex at 10.8 m from window start
C26 apex at 47.8 m from window start


corner,25,26,25,26,25,26,25,26
lap,5,5,10,10,21,21,23,23
entry_from_apex,-8.10,-24.10,-6.30,-25.30,-9.30,-26.20,-10.50,-26.20
peak_brake_bar,0.11,7.12,0.00,6.52,0.06,2.27,0.01,5.71
brake_total_time,0.00,0.92,0.00,0.74,0.00,0.45,0.00,0.45
coasting_time,0.14,0.58,0.00,0.52,0.00,1.24,0.00,1.07
coasting_dist,1.20,5.10,0.00,4.50,0.00,11.70,0.00,12.00
min_speed,8.43,8.85,9.29,8.36,7.78,8.25,7.92,8.40
min_speed_from_apex,-3.60,-0.10,-6.30,-0.10,-9.30,-4.30,-10.50,4.40
pickup_from_apex,-4.20,-1.60,NaN,-1.60,NaN,-5.20,NaN,-1.60
max_exit_throttle,0.45,0.26,0.50,0.30,0.29,0.23,0.46,0.24


* Metrics:
  - Lap 23 is 0.27 s faster than lap 10 in the snail. In C26 it brakes the shortest (0.45 s) and lighter than lap 10 (5.71 bar against 6.52 bar), and both laps start their C26 entry at almost the same point (26.2 m and 25.3 m before the apex), so their C26 times can be compared: 2.66 s for lap 23 against 3.21 s for lap 10. That is where lap 23 makes its time.
  - Lap 23 also coasts the most in C26 with lap 21, 12 m (1.07 s) against 4.5 m for lap 10. So here coasting does not cost time, lap 23 lifts early, brakes short and lets the car roll with more speed towards C26.
  - In C25 lap 10 is the best, it never lifts off completely, its minimum speed is 9.29 m/s against 7.92 m/s for lap 23, and it exits C25 the fastest at 11.78 m/s. Lap 10 also has a high rear wheel slip at the C25 exit (28.9%) with the throttle at 0.5, the same pattern as the fast laps in C24.
  - Lap 5 loses 0.14 s and the loss is in C25. It enters the slowest (7.26 m/s), coasts for 1.2 m before C25 and exits C25 at 10.92 m/s, the second slowest. In C26 it actually does the best, it has the highest minimum speed (8.85 m/s) and the highest exit speed (9.43 m/s), even though it brakes the hardest (7.12 bar).
  - Lap 21 loses 0.47 s. It is the slowest through C25 with a minimum speed of 7.78 m/s and an exit of 10.06 m/s, it brakes the lightest in C26 (2.27 bar) and coasts for 11.7 m (1.24 s), and it picks up the throttle the earliest in C26 (5.2 m before the apex) but only up to 0.23. The numbers for lap 21 look normal here, so the broken position data around C20 does not seem to affect the snail.
  - Correction to the plot reading: the minimum speed of lap 23 in C26 is 8.40 m/s, reached 4.4 m after the apex, so it is almost the same as lap 10 (8.36 m/s) and not higher.

In [40]:
# hairpin
plot_group("C27")

In [43]:
from src.config import DISTANCE_GRID_STEP_M
def check_group(name: str):
    w = gt[gt["group"] == name].iloc[0]
    s0, span = float(w["s_start"]), np.mod(float(w["s_end"]) - float(w["s_start"]), length)
    rows = []
    for lap, lg in grid[grid["lap"].isin(laps_used)].groupby("lap"):
        u = lg.loc[np.mod(lg["s"] - s0, length) <= span, U_COL]
        rows.append({"lap": lap, "time_from_speed": (DISTANCE_GRID_STEP_M / u).sum()})
    out = gt[gt["group"] == name][["lap", "time"]].merge(pd.DataFrame(rows), on="lap")
    out["diff"] = out["time"] - out["time_from_speed"]
    return out.sort_values("time")


check_group("snail")

,lap,time,time_from_speed,diff
7,9,5.115958,5.277643,-0.161685
20,23,5.197010,5.621265,-0.424256
5,7,5.214660,5.513220,-0.298560
10,12,5.297428,5.243464,0.053965
6,8,5.326161,5.345214,-0.019053
9,11,5.389676,5.356124,0.033552
11,13,5.409774,5.410899,-0.001124
15,18,5.451884,5.376901,0.074983
13,16,5.461701,5.482189,-0.020488
8,10,5.466336,5.487994,-0.021658


The times of some laps are not reliable. Lap 3 came out as the fastest lap, but on the plot it has the lowest speed almost everywhere, the lowest speed at the apex and the lowest exit speed, so it cannot be the fastest. To check this we compared the time of each lap in the window with the time worked out from the speed alone (adding up 0.3 m / speed over every point of the window). For most laps the two times agree within around 0.1 s, but for laps 7, 9 and 23 the time is around 0.8 to 1 s longer than what the speed says, even though they have the highest average speeds, and for lap 3 it is around 0.26 s shorter. This is the same problem as lap 21 in C20, after a gap in the data the position (s) stalls and then catches up, so the time ends up in the wrong place. These laps are not reliable in C27, so we leave them out and take the fastest and slowest laps from the remaining ones.

In [45]:
gt.loc[(gt["group"] == "C27") & gt["lap"].isin([3, 7, 9, 23]), "gap"] = True   # t_lap broken on these laps around C27 (time vs speed check)
picks = pick_laps(gt, ref)
plot_group("C27").show()

Comparison of laps 4, 5 and 21 with lap 10 in C27:
* Lap 10 (ref): the driver enters at 9.1 m/s with the throttle at 0.28, lifts off and starts braking at around 1 m. The braking goes up to around 4.7 bar at 2.5 m and then comes down gradually, staying at around 1.5 bar until around 6.5 m before releasing, so it is one smooth brake with a long and gentle release while he turns in. The steering builds up smoothly to around 91 deg at the apex. He gets on the throttle at around 9.3 m, before the apex, but only lightly (around 0.18) and keeps it steady until the apex, and the speed stays steady at around 7 m/s, the highest of all at the apex. After the apex he builds the throttle up gradually to 0.64 and exits at around 11.9 m/s.
* Lap 4 (fastest): the driver enters the fastest of all, at 9.8 m/s, and is already braking at the start of the window. The braking goes up quickly to around 4.5 bar at 1.2 m, then comes down to around 1 bar and goes up again a little to around 2 bar at 7.5 m before releasing at around 9 m, then he coasts for about 2.5 m. He turns in the quickest of all, the steering reaches around 82 deg already at 4 m. He carries more speed than lap 10 in the first half of the corner, but his speed at the apex is lower, around 6.5 m/s. He gets on the throttle at around 11.8 m and has more throttle at the apex than lap 10 (around 0.29 against 0.18), then builds it up to 0.68 and exits at around 11.7 m/s, almost the same as lap 10. So his time is gained in the first half of the corner, mostly from arriving faster.
* Lap 5: the driver enters at 9.4 m/s and brakes the hardest but the shortest of all, up to around 5.1 bar at 2.4 m, and releases at around 5.3 m. He then gets on the throttle early, at around 7.5 m, up to 0.2, but at the same time the steering opens from around 85 to 56 deg, then he lifts off again at around 10 m, tightens the steering back to around 97 deg and coasts until around 13.7 m. So he goes throttle, lift, throttle before the apex with a steering correction, which looks like he got on the throttle too early and had to correct his line. The speed drops to around 6.5 m/s at the apex because of the coast. After the apex he gets on the throttle quickly up to 0.4 and exits the fastest of all at around 12 m/s.
* Lap 21: the driver enters the slowest, at 8.8 m/s, and brakes the lightest and shortest of all, up to around 3.7 bar, releasing at around 4.4 m. He then coasts for around 5 m before getting on the throttle at around 9.2 m, and he is the slowest through the whole first part of the corner, down to around 6.5 m/s between 10 and 12 m. After the apex his throttle builds up like the others, up to 0.7 at the exit, but he exits the slowest at around 11.45 m/s.

* Additional notes:
  - Compared to C24, lap 10 does not overslow here. It brakes once, smoothly, and keeps a steady speed through the apex, while in C24 it dropped to 4.4 m/s at full lock.
  - In both hairpins the stint 2 driver (lap 16 in C24, lap 21 here) brakes lightly and coasts before the apex, and is the slowest in the first part of the corner.
  - In both hairpins the slowest stint 1 lap shows uneven inputs, the uneven braking of lap 3 in C24 and the throttle, lift, throttle with a steering correction of lap 5 here.
  - The fastest laps in both hairpins get the car turned early and use more throttle at or right after the apex than lap 10.

In [46]:
group_metrics("C27")

C27 apex at 14.1 m from window start


corner                    27                     
lap                       4      5      10     21
entry_from_apex       -14.10 -13.80 -13.20 -13.50
peak_brake_bar          4.51   5.09   4.76   3.78
brake_total_time        0.93   0.58   0.62   0.48
coasting_time           0.40   0.70   0.28   0.73
coasting_dist           3.00   5.40   2.10   5.10
min_speed               6.47   6.48   6.83   6.44
min_speed_from_apex    -1.80   0.00  -4.50  -3.00
pickup_from_apex       -2.10  -0.30  -4.80  -4.20
max_exit_throttle       0.68   0.64   0.64   0.70
exit_speed             11.69  11.98  11.91  11.46
steer_corrections       2.00   4.00   1.00   2.00
exit_slip_pct           4.29   5.58   5.89   3.93
corner_time             3.15   3.31   3.16   3.39
speed_at_window_start   9.82   9.43   9.07   8.80
group_time              3.15   3.35   3.27   3.45
group_delta_vs_ref     -0.12   0.08   0.00   0.18

* Metrics:
  - Lap 4 is 0.12 s faster than lap 10 in C27. It arrives the fastest of all (9.82 m/s against 9.07 m/s for lap 10), which matches the time gained in the first half of the corner. It brakes the longest (0.93 s), which matches the two step braking on the plot, and has a lower minimum speed than lap 10 (6.47 m/s against 6.83 m/s).
  - Lap 10 has the highest minimum speed (6.83 m/s, reached 4.5 m before the apex), picks up the throttle the earliest (4.8 m before the apex) and is the smoothest of all with only 1 steering correction.
  - Lap 5 loses only 0.08 s. It brakes the hardest (5.09 bar), coasts for 5.4 m and has 4 steering corrections, the most of all, which matches the throttle, lift, throttle and the steering correction before the apex. Its minimum speed is right at the apex (6.48 m/s), but it exits the fastest of all at 11.98 m/s.
  - Lap 21 loses 0.18 s. It brakes the lightest (3.78 bar) and the shortest (0.48 s), coasts for 5.1 m, and has the lowest minimum speed (6.44 m/s) and the lowest exit speed (11.46 m/s), even though it uses the most throttle on the exit (0.70).
  - The differences in C27 are small compared to the other corners, all the exit speeds are between 11.5 and 12 m/s, so C27 itself is not where most of the time is lost.

## Recommendations per corner for each driver


C27 to C1
The exit of C27 leads straight to C1, so whatever is lost on the C27 exit is carried all the way to the C1 braking point. Lap 10 exits C27 at 11.9 m/s and reaches 15.2 m/s at the start of the C1 window. Getting on the throttle early and progressively out of C27 is the first step for a good C1.

C1
* Driver 1: brake harder and shorter, do not coast between the brake and the throttle, and get on the throttle before the apex. Up to 0.85 s.
* Driver 2: arrive faster from C27, brake instead of just lifting, and commit to the throttle on the exit (it never goes above 0.23). Up to 1.57 s.
* Best lap (lap 10): build the throttle up more smoothly after the apex to stop the rear from sliding.

Esses (C2 to C5)
* Driver 1: brake lighter (around 9 to 10 bar instead of 16) and release before the C2 apex, carry more speed into C2 and do not brake again for C3. Up to 0.65 s for the slowest lap, around 0.2 s for the best lap.
* Driver 2: do not coast on the approach, stay on the throttle longer and brake later, and use more throttle on the exit. Driver 2 already has the fastest lap here (lap 22), so it is about doing it every lap. Up to 0.76 s.

C20
* Driver 1: one firm brake instead of a two step brake, and turn in once instead of turning in early and correcting. Up to 0.68 s.
* Driver 2: do not coast on the approach, stay on the throttle longer and brake later and harder. Up to 0.70 s.

C24
* Driver 1: brake hard at the start and then release progressively, keep the brake pressure smooth. On the best lap, release the brake before reaching full lock so the speed does not drop to 4.4 m/s, and build the throttle up more progressively to reduce the wheelspin on the exit. Up to 0.41 s, around 0.2 s for the best lap.
* Driver 2: brake earlier and harder instead of coasting before braking, and do not open the steering before the apex. Up to 0.19 s.

Snail (C25 and C26)
* Driver 1: do not lift off completely before C25, then brake lighter and shorter for C26 and do not drag the brake after it. Up to 0.14 s.
* Driver 2: use more throttle through C25, brake firmer and shorter for C26 with less coasting, and get on the throttle earlier and harder on the C26 exit. Up to 0.47 s.

C27
* Driver 1: wait until the car is pointed before getting on the throttle, so there is no throttle, lift, throttle before the apex, then commit to the throttle from the apex. Up to 0.08 s. The fastest lap here (lap 4) gains 0.12 s on the best lap mostly by arriving faster from C26, so a better C26 exit also helps C27.
* Driver 2: brake later and harder instead of a light brake and a long coast, and carry more speed in the first half of the corner. Up to 0.18 s.

Overall
* Driver 1 is fast but uneven, the braking often goes up and down and he sometimes gets on the throttle too early and has to correct.
* Driver 2 loses most of the time before the corners, with long coasting on the approach and light braking, and is often slow to commit to the throttle on the exit.